# Plank Counter Pose Corrector
Converted from the latest Python script.


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import os
from pathlib import Path
from typing import Tuple, Optional, List

# --- CONFIGURATION ---
BASE_DIR = Path(__file__).resolve().parent
VIDEO_PATH = str(BASE_DIR / "Plank_vids" / "plank1.webm")
WINDOW_NAME = "Plank Analysis"
SMOOTHING_FACTOR = 0.5

# Colors (BGR format)
COLOR_GOOD = (0, 255, 0)  # Green
COLOR_OKAY = (0, 165, 255)  # Orange
COLOR_BAD = (0, 0, 255)  # Red

mp_pose = mp.solutions.pose


def calculate_angle(a: List, b: List, c: List) -> float:
    """Calculates angle between three points."""
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(
        a[1] - b[1], a[0] - b[0]
    )
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle
    return angle


def get_plank_status(
    hip_angle: float, hip_y: float, expected_hip_y: float, knee_angle: float
) -> Tuple[str, Tuple[int, int, int]]:
    """
    Revised Logic:
    - 135 to 155 is PERFECT (Safe zone).
    - > 155 is BAD (Hips sagging/too straight, risking back pain).
    - < 135 is BAD (Hips too high).
    """

    # 1. Check for bent knees (Threshold 150)
    if knee_angle < 150:
        return "Keep Knees Straight!", COLOR_BAD

    # 2. Form Logic

    # PERFECT ZONE (Core engaged, slightly piked)
    if 143 <= hip_angle <= 163:
        return "Perfect Form", COLOR_GOOD

    # HIPS DROPPING (Angle getting too straight/wide, > 155)
    elif hip_angle > 155:
        return "Hips Too Low! (Lift Hips)", COLOR_BAD

    # HIPS TOO HIGH (Piking too much, < 135)
    else:
        return "Hips Too High!", COLOR_BAD


def main() -> None:
    if not os.path.exists(VIDEO_PATH):
        print(f"Error: Video file not found at {VIDEO_PATH}")
        return

    cap = cv2.VideoCapture(VIDEO_PATH)

    prev_hip: Optional[float] = None
    prev_knee: Optional[float] = None
    prev_elbow: Optional[float] = None

    total_time_held: float = 0.0
    last_frame_time: float = time.time()

    with mp_pose.Pose(
        min_detection_confidence=0.7, min_tracking_confidence=0.7
    ) as pose:

        while cap.isOpened():
            ret, frame = cap.read()

            current_time = time.time()
            dt = current_time - last_frame_time
            last_frame_time = current_time

            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue

            frame = cv2.resize(frame, (900, 600))
            final_image = np.zeros((600, 1200, 3), dtype=np.uint8)
            final_image[0:600, 0:900] = frame

            h, w, _ = frame.shape
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)

            status = "Waiting..."
            color = (255, 255, 255)

            if results.pose_landmarks:
                lm = results.pose_landmarks.landmark

                # --- AUTO-DETECT SIDE ---
                if lm[11].visibility > lm[12].visibility:
                    shoulder_pt = [lm[11].x * w, lm[11].y * h]
                    elbow_pt = [lm[13].x * w, lm[13].y * h]
                    wrist_pt = [lm[15].x * w, lm[15].y * h]
                    hip_pt = [lm[23].x * w, lm[23].y * h]
                    knee_pt = [lm[25].x * w, lm[25].y * h]
                    ankle_pt = [lm[27].x * w, lm[27].y * h]
                else:
                    shoulder_pt = [lm[12].x * w, lm[12].y * h]
                    elbow_pt = [lm[14].x * w, lm[14].y * h]
                    wrist_pt = [lm[16].x * w, lm[16].y * h]
                    hip_pt = [lm[24].x * w, lm[24].y * h]
                    knee_pt = [lm[26].x * w, lm[26].y * h]
                    ankle_pt = [lm[28].x * w, lm[28].y * h]

                # --- CALCULATE ANGLES ---
                raw_hip = calculate_angle(shoulder_pt, hip_pt, ankle_pt)
                raw_knee = calculate_angle(hip_pt, knee_pt, ankle_pt)
                raw_elbow = calculate_angle(shoulder_pt, elbow_pt, wrist_pt)

                # --- SMOOTHING ---
                if prev_hip is None:
                    prev_hip, prev_knee, prev_elbow = raw_hip, raw_knee, raw_elbow

                curr_hip = (
                    prev_hip * (1 - SMOOTHING_FACTOR) + raw_hip * SMOOTHING_FACTOR
                )
                curr_knee = (
                    prev_knee * (1 - SMOOTHING_FACTOR) + raw_knee * SMOOTHING_FACTOR
                )
                curr_elbow = (
                    prev_elbow * (1 - SMOOTHING_FACTOR) + raw_elbow * SMOOTHING_FACTOR
                )

                prev_hip, prev_knee, prev_elbow = curr_hip, curr_knee, curr_elbow

                # --- LOGIC ---
                if shoulder_pt[0] != ankle_pt[0]:
                    m = (ankle_pt[1] - shoulder_pt[1]) / (ankle_pt[0] - shoulder_pt[0])
                    expected_hip_y = shoulder_pt[1] + m * (hip_pt[0] - shoulder_pt[0])
                else:
                    expected_hip_y = (shoulder_pt[1] + ankle_pt[1]) / 2

                status, color = get_plank_status(
                    curr_hip, hip_pt[1], expected_hip_y, curr_knee
                )

                # --- TIMER UPDATE ---
                if color != COLOR_BAD:
                    total_time_held += dt

                # --- VISUALIZATION ---
                cv2.line(
                    final_image,
                    tuple(np.array(shoulder_pt, int)),
                    tuple(np.array(hip_pt, int)),
                    (255, 255, 255),
                    2,
                )
                cv2.line(
                    final_image,
                    tuple(np.array(hip_pt, int)),
                    tuple(np.array(ankle_pt, int)),
                    (255, 255, 255),
                    2,
                )
                cv2.line(
                    final_image,
                    tuple(np.array(shoulder_pt, int)),
                    tuple(np.array(elbow_pt, int)),
                    (255, 255, 255),
                    2,
                )
                cv2.line(
                    final_image,
                    tuple(np.array(elbow_pt, int)),
                    tuple(np.array(wrist_pt, int)),
                    (255, 255, 255),
                    2,
                )

                cv2.circle(
                    final_image, tuple(np.array(elbow_pt, int)), 6, (255, 255, 255), -1
                )
                cv2.circle(final_image, tuple(np.array(hip_pt, int)), 8, color, -1)

                # --- STATS PANEL ---
                cv2.putText(
                    final_image,
                    "TIME HELD:",
                    (920, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 255),
                    2,
                )
                cv2.putText(
                    final_image,
                    f"{total_time_held:.1f} s",
                    (920, 100),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5,
                    color,
                    3,
                )

                cv2.putText(
                    final_image,
                    f"HIP: {int(curr_hip)}",
                    (920, 180),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2,
                )
                cv2.putText(
                    final_image,
                    f"KNEE: {int(curr_knee)}",
                    (920, 230),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 255),
                    1,
                )
                cv2.putText(
                    final_image,
                    f"ELBOW: {int(curr_elbow)}",
                    (920, 280),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 255),
                    1,
                )

                cv2.putText(
                    final_image,
                    status,
                    (920, 380),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    color,
                    2,
                )

            cv2.imshow(WINDOW_NAME, final_image)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
